# Plate processing

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plate_utils import load_plate, normalize
%matplotlib inline
CONDITION_ORDER = ["DMSO", "drugA_1uM", "drugA_10uM"]

In [ ]:
DATA_DIR = r"C:\Users\sam\Desktop\plate_assay\data"
OUT_DIR = r"C:\Users\sam\Desktop\plate_assay\results"
PLATE = "plate1"   # change to plate2 and run again

In [ ]:
cells, layout = load_plate(DATA_DIR, PLATE)
print(layout["plate"], len(cells), "cells")
cells.head()

In [ ]:
MIN_AREA = 30
cells = cells[cells["area"] >= MIN_AREA]

In [ ]:
norm = normalize(cells, layout["background"])
norm.to_csv(os.path.join(OUT_DIR, f"normalized_{PLATE}.csv"), index=False)
norm.groupby("condition")["fold_change"].median()

In [ ]:
# bootstrap 95% CI of the mean fold change per condition
N_BOOT = 5000
rows = []
for cond in CONDITION_ORDER:
    fc = norm.loc[norm["condition"] == cond, "fold_change"].to_numpy()
    means = []
    for i in range(N_BOOT):
        means.append(np.random.choice(fc, size=len(fc), replace=True).mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    rows.append({"condition": cond, "mean_fc": fc.mean(), "ci_low": lo, "ci_high": hi,
                 "n_cells": len(fc)})
boot = pd.DataFrame(rows)
boot.to_csv(os.path.join(OUT_DIR, f"bootstrap_{PLATE}.csv"), index=False)
boot